# sw2023 — JSS Replication Notebook

Lee, C. (2026). *sw2023: Nonparametric Multiple-Output Stochastic Frontier Analysis in Python.*  
Manuscript submitted for publication.

This notebook reproduces all numerical results and figures reported in the manuscript.
It mirrors `replication.py` section by section, with inline outputs and plots.

| Section | Content |
|---|---|
| 0 | Setup — install, imports, versions |
| 1 | Tables 1 & 2 — Monte Carlo validation |
| 2 | Table 3 — Simulation-based illustration (Section 6) |
| 3 | Table 4 — Norwegian agricultural panel (Section 7) |
| 4 | Figures 1 & 2 — Manuscript figures |

**Runtime:** \< 10 minutes with default settings (`N_SIMS = 20`).  
Set `N_SIMS = 100` and `N_LIST = [100, 200, 400, 800]` in Section 1 to reproduce the paper exactly (~60 min).

## 0. Setup

In [ ]:
# Install sw2023 from the accompanying tarball (JSS submission version)
# Uncomment one of the lines below:
# !pip install sw2023-0.3.2.tar.gz    # from tarball (recommended for exact replication)
# !pip install sw2023==0.3.2           # from PyPI

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import gaussian_kde

import sw2023, scipy, matplotlib

print(f'sw2023     : {sw2023.__version__}')
print(f'Python     : {sys.version.split()[0]}')
print(f'numpy      : {np.__version__}')
print(f'scipy      : {scipy.__version__}')
print(f'pandas     : {pd.__version__}')
print(f'matplotlib : {matplotlib.__version__}')

In [ ]:
from sw2023 import (SW2023Model, PanelSW2023,
                    bootstrap_sw, test_r3_significance)
from sw2023.tests.monte_carlo import run_imse_grid, print_imse_comparison

# JSS plot style
plt.rcParams.update({
    'font.family'      : 'serif',
    'font.size'        : 10,
    'axes.labelsize'   : 10,
    'axes.titlesize'   : 10,
    'legend.fontsize'  : 9,
    'xtick.labelsize'  : 9,
    'ytick.labelsize'  : 9,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
})
COLORS = {
    'silverman'   : '#2166ac',
    'loocv_scalar': '#d6604d',
    'loocv'       : '#1a9850',
}

HERE = os.path.dirname(os.path.abspath('replication.ipynb'))
print('Setup complete.')

## 1. Tables 1 & 2 — Monte Carlo Validation (Section 5)

Reproduces IMSE ratios relative to Simar & Wilson (2023, JBES) Table F.1.

- **Table 1**: scalar LOO-CV bandwidth → `mc_imse_results.csv`
- **Table 2**: product LOO-CV bandwidth → `mc_imse_product.csv`

**DGP**: Marsaglia (1972) sphere, $\sigma_\eta = 0.5$, $\rho = 0$.  
Set `N_SIMS = 100` and `N_LIST = [100, 200, 400, 800]` to reproduce the paper exactly.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
N_SIMS = 20                        # paper: 100  (set to 100 for exact replication)
N_LIST = [100, 200, 400]           # paper: [100, 200, 400, 800]
PQ_LIST = [(1,1), (1,2), (2,1), (2,2)]

if N_SIMS < 100:
    print(f'Note: N_SIMS={N_SIMS} (quick mode). Set N_SIMS=100, N_LIST=[100,200,400,800] for paper values.')

In [ ]:
# ── Table 1: Scalar LOO-CV ─────────────────────────────────────────────────
np.random.seed(2023)
csv_scalar = os.path.join(HERE, 'mc_imse_results.csv')

df_t1 = run_imse_grid(
    pq_list=PQ_LIST, n_list=N_LIST, rho_list=[0.0],
    n_sims=N_SIMS, bandwidth_method='loocv',
    out_csv=csv_scalar, verbose=True,
)
print('\n-- Table 1: Scalar LOO-CV (live run) --')
print_imse_comparison(df_t1)

In [ ]:
# ── Table 1: Pre-computed reference (n_sims=100, paper values) ───────────────
if os.path.exists(csv_scalar):
    print('-- Table 1 (pre-computed, n_sims=100) --')
    print(pd.read_csv(csv_scalar).to_string(index=False))

In [ ]:
# ── Table 2: Product LOO-CV ────────────────────────────────────────────────
np.random.seed(2023)
csv_product = os.path.join(HERE, 'mc_imse_product.csv')

df_t2 = run_imse_grid(
    pq_list=PQ_LIST, n_list=N_LIST, rho_list=[0.0],
    n_sims=N_SIMS, bandwidth_method='loocv',
    out_csv=csv_product, verbose=True,
)
print('\n-- Table 2: Product LOO-CV (live run) --')
print_imse_comparison(df_t2)

In [ ]:
if os.path.exists(csv_product):
    print('-- Table 2 (pre-computed, n_sims=100) --')
    print(pd.read_csv(csv_product).to_string(index=False))

## 2. Table 3 — Simulation-Based Illustration (Section 6)

**DGP**: Marsaglia sphere, $p=q=2$, $n=200$, $\sigma_\eta=0.5$, $\rho=1.0$.  
Reproduces all numerical outputs in Section 6 of the manuscript.

In [ ]:
# ── DGP ───────────────────────────────────────────────────────────────────────
np.random.seed(42)
n, p, q = 200, 2, 2

def sphere(n, dim):
    X = np.random.randn(n, dim)
    return X / np.linalg.norm(X, axis=1, keepdims=True)

pts = np.abs(sphere(n, p + q))
X_sim, Y_sim = pts[:, :p], pts[:, p:]
eta_sim = np.abs(np.random.randn(n)) * 0.5
eps_sim = np.random.randn(n) * 0.5
Y_sim   = Y_sim * np.clip(1 - eta_sim * 0.3 + eps_sim * 0.1, 0.5, 1.5)[:, None]
print(f'Data shape: X={X_sim.shape}, Y={Y_sim.shape}')

### 2.1 Cross-Sectional Model

In [ ]:
m_sim = SW2023Model(X_sim, Y_sim, direction='mean', method='HMS',
                    bandwidth_method='silverman')
m_sim.fit(verbose=True)
print(m_sim)   # __repr__

In [ ]:
print('-- Table 3, Row 1: Cross-sectional model --')
print(f'  Mean efficiency : {m_sim.efficiency_.mean():.4f}   (paper: 0.5311)')
print(f'  Std  efficiency : {m_sim.efficiency_.std():.4f}   (paper: 0.2538)')
print(f'  Mean sigma_eta  : {m_sim.sigma_eta_.mean():.4f}   (paper: 1.0551)')
print(f'  Wrong skewness  : {(m_sim.r3_>0).mean()*100:.1f}%')

### 2.2 Diagnostic Plots

In [ ]:
fig_diag = m_sim.plot_diagnostics(figsize=(12, 9))
plt.show()

### 2.3 Asymptotic Confidence Intervals

In [ ]:
ci_sim = m_sim.confint_asymptotic(alpha=0.05)
ci_sim.summary()
print(f'\n  Mean SE(phi_hat)        : {ci_sim.se_phi.mean():.4f}   (paper: 0.3745)')
print(f'  Mean CI lower (phi_hat) : {ci_sim.phi_hat_ci[:,0].mean():.4f}   (paper: -0.0064)')
print(f'  Mean CI upper (phi_hat) : {ci_sim.phi_hat_ci[:,1].mean():.4f}   (paper:  1.4618)')

### 2.4 Bootstrap Confidence Intervals

In [ ]:
# Via SW2023Model.bootstrap() method (B=199, seed fixed for reproducibility)
boot_sim = m_sim.bootstrap(B=199, seed=2023, verbose=True)
boot_sim.summary()

In [ ]:
width = (boot_sim.phi_hat_ci[:,1] - boot_sim.phi_hat_ci[:,0]).mean()
lo_b, hi_b = boot_sim.eff_mean_ci
print(f'  Mean CI width (frontier) : {width:.4f}   (paper: 2.0043)')
print(f'  Mean eff 95% CI          : [{lo_b:.4f}, {hi_b:.4f}]   (paper: [0.4661, 0.5757])')

### 2.5 Wild Bootstrap Significance Test

In [ ]:
# H0: E(eps^3 | Z) = const  (uniform inefficiency)
# seed=2023, B=999 for reproducibility
tres = test_r3_significance(X_sim, Y_sim, B=999, seed=2023, verbose=False)
tres.summary()
print(f'\n  Test statistic T : {tres.statistic:.4f}   (paper: 0.0397)')
print(f'  p-value          : {tres.p_value:.4f}   (seed=2023, B=999)')

## 3. Table 4 — Norwegian Agricultural Panel (Section 7)

Data: `norway_for_python.csv` — Kumbhakar, Wang & Horncastle (2015).  
$n = 2{,}729$ farm-year observations, 6 inputs, 4 outputs, years 1993–2002.

In [ ]:
data_path = os.path.join(HERE, 'norway_for_python.csv')
if not os.path.exists(data_path):
    print('[SKIP] norway_for_python.csv not found.')
    print('       Place it in the same directory as this notebook.')
else:
    np.random.seed(2023)
    df_no = (pd.read_csv(data_path)
               .sort_values(['farmid','year'])
               .reset_index(drop=True))
    X_no = df_no[['x1','x2','x3','x4','x5','x6']].values
    Y_no = df_no[['y1','y2','y3','y4']].values
    print(f'Loaded: {df_no.shape[0]} obs, {X_no.shape[1]} inputs, {Y_no.shape[1]} outputs')
    print(f'Years : {df_no.year.min()}–{df_no.year.max()}')

### 3.1 Cross-Sectional Model (paper Table 3)

In [ ]:
m_no = SW2023Model(X_no, Y_no, method='HMS', bandwidth_method='silverman')
m_no.fit(verbose=True)
print(m_no)

In [ ]:
eff_no = m_no.efficiency_
print('-- Table 4a: Cross-sectional summary (paper Table 3) --')
print(f'  Mean efficiency   : {eff_no.mean():.3f}   (paper: 0.813)')
print(f'  Median efficiency : {np.median(eff_no):.3f}   (paper: 0.831)')
print(f'  Std  efficiency   : {eff_no.std():.3f}   (paper: 0.131)')
print(f'  Min  efficiency   : {eff_no.min():.3f}   (paper: 0.251)')
print(f'  Wrong-skewness    : {(m_no.r3_>0).mean()*100:.1f}%   (paper: 50.6%)')

In [ ]:
fig_no_diag = m_no.plot_diagnostics()
plt.show()

### 3.2 Four-Component Panel Model (paper Table 4)

In [ ]:
m_panel = PanelSW2023(X_no, Y_no,
                      df_no['farmid'].values,
                      df_no['year'].values,
                      method='HMS')
m_panel.fit(verbose=True)

In [ ]:
df_res = df_no[['farmid','year']].copy()
df_res['cs_eff'] = m_no.efficiency_
df_res['te']     = m_panel.eff_transient_
df_res['pe']     = m_panel.eff_persistent_
yearly = df_res.groupby('year')[['cs_eff','te','pe']].mean()

print('-- Table 4b: Yearly mean efficiency (paper Table 4) --')
print(f"  {'Year':>4}  {'CS':>6}  {'TE':>6}  {'PE':>6}")
print('  ' + '-'*26)
for yr, row in yearly.iterrows():
    print(f"  {yr:>4}  {row['cs_eff']:>6.3f}  {row['te']:>6.3f}  {row['pe']:>6.3f}")
print('  ' + '-'*26)
print(f"  {'Mean':>4}  {yearly['cs_eff'].mean():>6.3f}  "
      f"{yearly['te'].mean():>6.3f}  {yearly['pe'].mean():>6.3f}")
print(f"  {'Paper':>4}  {'0.813':>6}  {'0.961':>6}  {'0.978':>6}")

## 4. Figures — Manuscript Figures 1 & 2

- **Figure 1**: Synthetic data — frontier estimates & efficiency distributions  
  (three bandwidth methods: Silverman, scalar LOO-CV, product LOO-CV)
- **Figure 2**: Norwegian data — efficiency distributions & yearly trend

### 4.1 Figure 1 — Synthetic Data (p=1, q=1, n=300)

In [ ]:
np.random.seed(2023)
n_fig = 300
x_raw    = np.sort(np.random.uniform(1, 5, n_fig))
phi_true = 2.0 * np.log(x_raw)
v_fig    = np.random.normal(0, 0.20, n_fig)
u_fig    = np.abs(np.random.normal(0, 0.30, n_fig))
y_fig    = np.exp(phi_true - u_fig + v_fig)

X_fig = x_raw.reshape(-1, 1)
Y_fig = y_fig.reshape(-1, 1)

results_fig1 = {}
for bw in ['silverman', 'loocv_scalar', 'loocv']:
    m_f = SW2023Model(X_fig, Y_fig, method='HMS',
                      bandwidth_method=bw,
                      log_transform=True, standardize=True)
    m_f.fit(verbose=False)
    order = np.argsort(m_f.Z_[:, 0])
    results_fig1[bw] = {
        'z'  : m_f.Z_[:, 0][order],
        'phi': m_f.phi_hat_[order],
        'eff': m_f.efficiency_,
    }
    print(f'  {bw:15s}: mean_eff={m_f.efficiency_.mean():.4f}')

In [ ]:
fig1, axes1 = plt.subplots(1, 2, figsize=(10, 4))

# (a) Frontier estimates
ax = axes1[0]
for bw, label, ls in [
    ('silverman',   'Silverman',      '-'),
    ('loocv_scalar','LOO-CV scalar',  '--'),
    ('loocv',       'LOO-CV product', ':'),
]:
    ax.plot(results_fig1[bw]['z'], results_fig1[bw]['phi'],
            color=COLORS[bw], ls=ls, lw=1.8, label=label)
ax.set_xlabel(r'Projected input $Z$')
ax.set_ylabel(r'Estimated frontier $\hat{\varphi}(Z)$')
ax.set_title(r'(a) Frontier estimates  ($p=1,\,q=1,\,n=300$)')
ax.legend(frameon=False)

# (b) Efficiency distributions
ax2 = axes1[1]
x_grid = np.linspace(0, 1, 300)
for bw, label, ls in [
    ('silverman',   'Silverman',      '-'),
    ('loocv_scalar','LOO-CV scalar',  '--'),
    ('loocv',       'LOO-CV product', ':'),
]:
    eff = results_fig1[bw]['eff']
    kde = gaussian_kde(eff, bw_method=0.15)
    ax2.plot(x_grid, kde(x_grid), color=COLORS[bw], ls=ls, lw=1.8, label=label)
    ax2.axvline(eff.mean(), color=COLORS[bw], ls=ls, lw=0.8, alpha=0.6)
ax2.set_xlabel('Efficiency score')
ax2.set_ylabel('Density')
ax2.set_title(r'(b) Efficiency distributions  (vertical lines = means)')
ax2.legend(frameon=False)

fig1.tight_layout()
fig1.savefig(os.path.join(HERE, 'fig_synthetic_comparison.pdf'), bbox_inches='tight')
fig1.savefig(os.path.join(HERE, 'fig_synthetic_comparison.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig_synthetic_comparison.pdf/png')

### 4.2 Figure 2 — Norwegian Agricultural Panel

In [ ]:
loocv_csv = os.path.join(HERE, 'norway_loocv_comparison.csv')
if not os.path.exists(loocv_csv):
    print(f'[SKIP] {loocv_csv} not found.')
    print('       Run run_loocv_comparison.py first to generate this file.')
else:
    df_fig2 = pd.read_csv(loocv_csv)
    print(f'Loaded: {df_fig2.shape[0]} rows, columns: {list(df_fig2.columns)}')

In [ ]:
if os.path.exists(loocv_csv):
    labels_fig2 = {
        'eff_silverman': ('Silverman',      '-',  COLORS['silverman']),
        'eff_scalar'   : ('LOO-CV scalar',  '--', COLORS['loocv_scalar']),
        'eff_product'  : ('LOO-CV product', ':',  COLORS['loocv']),
    }
    fig2, axes2 = plt.subplots(1, 2, figsize=(10, 4))

    # (a) Efficiency distributions
    ax = axes2[0]
    x_grid = np.linspace(0, 1.05, 400)
    for col, (label, ls, color) in labels_fig2.items():
        vals = df_fig2[col].dropna().values
        kde  = gaussian_kde(vals, bw_method=0.08)
        ax.plot(x_grid, kde(x_grid), color=color, ls=ls, lw=1.8, label=label)
        ax.axvline(vals.mean(), color=color, ls=ls, lw=0.8, alpha=0.7)
    ax.set_xlabel('Efficiency score')
    ax.set_ylabel('Density')
    ax.set_title('(a) Efficiency distributions\nNorwegian farms, 1993–2002  ($n=2{,}729$)')
    ax.legend(frameon=False)
    ax.set_xlim(0.1, 1.05)

    # (b) Yearly trend
    ax2 = axes2[1]
    yearly2 = df_fig2.groupby('year')[
        ['eff_silverman','eff_scalar','eff_product',
         'TE_silverman','PE_silverman']].mean()
    years2 = yearly2.index.astype(int)
    for col, (label, ls, color) in labels_fig2.items():
        ax2.plot(years2, yearly2[col], color=color, ls=ls,
                 lw=1.8, marker='o', ms=4, label=label)
    ax2.plot(years2, yearly2['TE_silverman'], color='#999999',
             ls='-.', lw=1.4, marker='s', ms=3, label='TE (4-comp)')
    ax2.plot(years2, yearly2['PE_silverman'], color='#555555',
             ls='-.', lw=1.4, marker='^', ms=3, label='PE (4-comp)')
    ax2.set_xlabel('Year')
    ax2.set_ylabel('Mean efficiency')
    ax2.set_title('(b) Mean efficiency by year\n(bandwidth methods + panel)')
    ax2.set_xticks(years2)
    ax2.set_xticklabels(years2, rotation=45)
    ax2.legend(frameon=False, fontsize=8, ncol=2)
    ax2.set_ylim(0.5, 1.02)

    fig2.tight_layout()
    fig2.savefig(os.path.join(HERE, 'fig_norway_comparison.pdf'), bbox_inches='tight')
    fig2.savefig(os.path.join(HERE, 'fig_norway_comparison.png'), bbox_inches='tight', dpi=150)
    plt.show()
    print('Saved: fig_norway_comparison.pdf/png')

---
## Reproducibility Summary

| Item | Seed | Notes |
|---|---|---|
| Tables 1 & 2 (Monte Carlo) | `np.random.seed(2023)` | Set `N_SIMS=100`, `N_LIST=[100,200,400,800]` for paper |
| Table 3 (Simulation) | `np.random.seed(42)` | Exactly reproducible |
| Bootstrap CI (Section 6.3) | `seed=2023` | `B=199` |
| Significance test (Section 6.4) | `seed=2023` | `B=999` |
| Table 4 (Norway) | `np.random.seed(2023)` | Requires `norway_for_python.csv` |
| Figures 1 & 2 | `np.random.seed(2023)` | Require `norway_loocv_comparison.csv` for Fig 2 |

All results were generated with **sw2023 v0.3.2** on Python 3.10.